# 03 — Seed Operational Dimension Tables

Creates five static dimension tables directly in `silver_rti` as managed Delta tables.
No streaming or CDC is required — run once (or re-run with `mode="overwrite"` to refresh).

| Table | Contents |
|---|---|
| `Employees` | Plant personnel (operators, supervisors, maintenance techs) |
| `Shifts` | Day / Night / Swing shift definitions |
| `ShiftAssignments` | Employee ↔ shift ↔ process unit assignments |
| `EquipmentSpecs` | Manufacturer, model, year, nominal power per equipment |
| `OperatingLimits` | Design min/max, trip low/high per parameter per equipment |

**Prerequisite:** `silver_rti` must be set as the default lakehouse for this notebook.

## Employees

In [ ]:
from pyspark.sql import Row

employees = [
    # Train A operators
    Row(id=1,  name="Carlos Mendez",    role="Operator",          train_assigned="Train A", shift_preference="Day",   certifications="H2S Awareness, First Aid"),
    Row(id=2,  name="Ana Torres",       role="Operator",          train_assigned="Train A", shift_preference="Night", certifications="H2S Awareness, Confined Space"),
    Row(id=3,  name="Luis Herrera",     role="Operator",          train_assigned="Train A", shift_preference="Swing", certifications="H2S Awareness"),
    Row(id=4,  name="Maria Rios",       role="Operator",          train_assigned="Train A", shift_preference="Day",   certifications="H2S Awareness, First Aid"),
    # Train A supervisor
    Row(id=5,  name="Jorge Castillo",   role="Supervisor",        train_assigned="Train A", shift_preference="Day",   certifications="H2S Awareness, First Aid, Process Safety"),
    Row(id=6,  name="Patricia Vargas",  role="Supervisor",        train_assigned="Train A", shift_preference="Night", certifications="H2S Awareness, Process Safety"),
    # Train B operators
    Row(id=7,  name="Roberto Silva",    role="Operator",          train_assigned="Train B", shift_preference="Day",   certifications="H2S Awareness, First Aid"),
    Row(id=8,  name="Elena Morales",    role="Operator",          train_assigned="Train B", shift_preference="Night", certifications="H2S Awareness, Confined Space"),
    Row(id=9,  name="Diego Fuentes",    role="Operator",          train_assigned="Train B", shift_preference="Swing", certifications="H2S Awareness"),
    Row(id=10, name="Sofia Gutierrez",  role="Operator",          train_assigned="Train B", shift_preference="Day",   certifications="H2S Awareness, First Aid"),
    # Train B supervisor
    Row(id=11, name="Miguel Ortega",    role="Supervisor",        train_assigned="Train B", shift_preference="Day",   certifications="H2S Awareness, First Aid, Process Safety"),
    Row(id=12, name="Laura Medina",     role="Supervisor",        train_assigned="Train B", shift_preference="Night", certifications="H2S Awareness, Process Safety"),
    # Maintenance (cross-train)
    Row(id=13, name="Fernando Cruz",    role="Maintenance Tech",  train_assigned="Both",    shift_preference="Day",   certifications="H2S Awareness, Machinery, Instrumentation"),
    Row(id=14, name="Isabella Romero",  role="Maintenance Tech",  train_assigned="Both",    shift_preference="Night", certifications="H2S Awareness, Machinery"),
]

df_employees = spark.createDataFrame(employees)
df_employees.write.format("delta").mode("overwrite").saveAsTable("Employees")
print(f"Employees: {df_employees.count()} rows")

## Shifts

In [ ]:
shifts = [
    Row(id=1, name="Day",   start_hour=6,  end_hour=18, days_of_week="Mon-Sun"),
    Row(id=2, name="Night", start_hour=18, end_hour=6,  days_of_week="Mon-Sun"),
    Row(id=3, name="Swing", start_hour=6,  end_hour=18, days_of_week="Rotating 4-on-4-off"),
]

df_shifts = spark.createDataFrame(shifts)
df_shifts.write.format("delta").mode("overwrite").saveAsTable("Shifts")
print(f"Shifts: {df_shifts.count()} rows")

## ShiftAssignments

ProcessUnit IDs: 2=Train A, 8=Train B (train-level assignments for operators/supervisors).

In [ ]:
# unit_id: 2=Train A, 8=Train B
# shift_id: 1=Day, 2=Night, 3=Swing
shift_assignments = [
    Row(id=1,  employee_id=1,  shift_id=1, unit_id=2, date_start="2025-01-01", date_end="2025-12-31"),
    Row(id=2,  employee_id=2,  shift_id=2, unit_id=2, date_start="2025-01-01", date_end="2025-12-31"),
    Row(id=3,  employee_id=3,  shift_id=3, unit_id=2, date_start="2025-01-01", date_end="2025-12-31"),
    Row(id=4,  employee_id=4,  shift_id=1, unit_id=2, date_start="2025-01-01", date_end="2025-12-31"),
    Row(id=5,  employee_id=5,  shift_id=1, unit_id=2, date_start="2025-01-01", date_end="2025-12-31"),
    Row(id=6,  employee_id=6,  shift_id=2, unit_id=2, date_start="2025-01-01", date_end="2025-12-31"),
    Row(id=7,  employee_id=7,  shift_id=1, unit_id=8, date_start="2025-01-01", date_end="2025-12-31"),
    Row(id=8,  employee_id=8,  shift_id=2, unit_id=8, date_start="2025-01-01", date_end="2025-12-31"),
    Row(id=9,  employee_id=9,  shift_id=3, unit_id=8, date_start="2025-01-01", date_end="2025-12-31"),
    Row(id=10, employee_id=10, shift_id=1, unit_id=8, date_start="2025-01-01", date_end="2025-12-31"),
    Row(id=11, employee_id=11, shift_id=1, unit_id=8, date_start="2025-01-01", date_end="2025-12-31"),
    Row(id=12, employee_id=12, shift_id=2, unit_id=8, date_start="2025-01-01", date_end="2025-12-31"),
    Row(id=13, employee_id=13, shift_id=1, unit_id=2, date_start="2025-01-01", date_end="2025-12-31"),
    Row(id=14, employee_id=14, shift_id=2, unit_id=8, date_start="2025-01-01", date_end="2025-12-31"),
]

df_sa = spark.createDataFrame(shift_assignments)
df_sa.write.format("delta").mode("overwrite").saveAsTable("ShiftAssignments")
print(f"ShiftAssignments: {df_sa.count()} rows")

## EquipmentSpecs

ProcessUnit IDs:
- 3=A-Separator, 4=A-K100, 5=A-K200, 6=A-K300, 7=A-Meter  → parent_unit_id=2 (Train A)
- 9=B-Separator, 10=B-K100, 11=B-K200, 12=B-K300, 13=B-Meter → parent_unit_id=8 (Train B)

`parent_unit_id` links each equipment to its parent train, enabling the join path:
`SensorReadings → EquipmentSpecs → ShiftAssignments → Employees`

In [ ]:
equipment_specs = [
    # Separators — parent: Train A=2, Train B=8
    Row(equipment_id=3,  parent_unit_id=2, manufacturer="Exterran",          model="HZS-2400",    year_built=2018, nominal_power_kw=None,   compressor_type=None),
    Row(equipment_id=9,  parent_unit_id=8, manufacturer="Exterran",          model="HZS-2400",    year_built=2019, nominal_power_kw=None,   compressor_type=None),
    # Train A compressors
    Row(equipment_id=4,  parent_unit_id=2, manufacturer="Ariel Corporation", model="JGC/4",       year_built=2017, nominal_power_kw=1120.0, compressor_type="Reciprocating"),
    Row(equipment_id=5,  parent_unit_id=2, manufacturer="Ariel Corporation", model="JGC/6",       year_built=2019, nominal_power_kw=1490.0, compressor_type="Reciprocating"),
    Row(equipment_id=6,  parent_unit_id=2, manufacturer="Solar Turbines",    model="Mars 90",     year_built=2020, nominal_power_kw=9400.0, compressor_type="Centrifugal"),
    # Train B compressors
    Row(equipment_id=10, parent_unit_id=8, manufacturer="Ariel Corporation", model="JGC/4",       year_built=2018, nominal_power_kw=1120.0, compressor_type="Reciprocating"),
    Row(equipment_id=11, parent_unit_id=8, manufacturer="Ariel Corporation", model="JGC/6",       year_built=2020, nominal_power_kw=1490.0, compressor_type="Reciprocating"),
    Row(equipment_id=12, parent_unit_id=8, manufacturer="Solar Turbines",    model="Mars 90",     year_built=2021, nominal_power_kw=9400.0, compressor_type="Centrifugal"),
    # Metering
    Row(equipment_id=7,  parent_unit_id=2, manufacturer="Emerson",           model="Daniel 3416", year_built=2018, nominal_power_kw=None,   compressor_type=None),
    Row(equipment_id=13, parent_unit_id=8, manufacturer="Emerson",           model="Daniel 3416", year_built=2019, nominal_power_kw=None,   compressor_type=None),
]

df_specs = spark.createDataFrame(equipment_specs)
df_specs.write.format("delta").mode("overwrite").saveAsTable("EquipmentSpecs")
print(f"EquipmentSpecs: {df_specs.count()} rows")

## OperatingLimits

Design and trip setpoints for compressors (equipment_ids 4, 5, 6 / Train A; 10, 11, 12 / Train B).
Units match sensor `unit_of_measure` field:
- Pressure → bar g
- Temperature → °C
- Speed → RPM
- Power → kW

In [ ]:
def limits(equipment_id, parameter_type, design_min, design_max, trip_low, trip_high):
    return Row(
        equipment_id=equipment_id,
        parameter_type=parameter_type,
        design_min=float(design_min),
        design_max=float(design_max),
        trip_low=float(trip_low),
        trip_high=float(trip_high),
    )

# Reciprocating K100/K300 (JGC/4, JGC/6) — for eqp_ids 4, 5, 10, 11
# Centrifugal K300/K300 (Mars 90)           — for eqp_ids 6, 12

operating_limits = []

# ── Reciprocating compressors (4, 5, 10, 11) ─────────────────────────────────
for eid in [4, 5, 10, 11]:
    operating_limits += [
        limits(eid, "suction_pressure_barg",    1.0,  8.0,  0.5,   8.5),
        limits(eid, "discharge_pressure_barg",  15.0, 45.0, 12.0, 48.0),
        limits(eid, "suction_temp_c",           5.0,  55.0,  2.0,  60.0),
        limits(eid, "discharge_temp_c",         30.0, 135.0, 25.0, 145.0),
        limits(eid, "speed_rpm",                600.0, 1200.0, 500.0, 1250.0),
        limits(eid, "power_kw",                 200.0, 1200.0, 0.0,  1300.0),
    ]

# ── Centrifugal compressors (6, 12) ───────────────────────────────────────────
for eid in [6, 12]:
    operating_limits += [
        limits(eid, "suction_pressure_barg",    2.0,  10.0,  1.0,  11.0),
        limits(eid, "discharge_pressure_barg",  20.0, 60.0, 15.0,  65.0),
        limits(eid, "suction_temp_c",           5.0,  45.0,  2.0,  50.0),
        limits(eid, "discharge_temp_c",         30.0, 120.0, 25.0, 130.0),
        limits(eid, "speed_rpm",                3000.0, 12000.0, 2500.0, 12500.0),
        limits(eid, "power_kw",                 1000.0, 9500.0, 500.0, 10000.0),
    ]

df_limits = spark.createDataFrame(operating_limits)
df_limits.write.format("delta").mode("overwrite").saveAsTable("OperatingLimits")
print(f"OperatingLimits: {df_limits.count()} rows")

## Verify all tables

In [ ]:
for table in ["Employees", "Shifts", "ShiftAssignments", "EquipmentSpecs", "OperatingLimits"]:
    count = spark.table(table).count()
    print(f"{table:25s}: {count:3d} rows")